In [ ]:
from pathlib import Path
import sys


here = Path.cwd()
candidates = [here] + list(here.parents)
for p in candidates:
    if (p / "Code").is_dir():
        sys.path.insert(0, str(p))
        break
else:
    raise RuntimeError("Couldn't find a 'Code' folder in this project.")

from autograd import grad, elementwise_grad
import autograd.numpy as np 
from sklearn.utils import resample
from copy import copy
from Code.activations import sigmoid, identity, derivate
from Code.scheduler import Scheduler, Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from Code.ffnn2 import NeuralNetwork as FFNN

In [5]:


# --- Cost Function (Mean Squared Error) ---
def CostOLS(target):
    def func(X):
        return np.mean((target - X)**2)
    return func


# --- Data Generation Helper ---
# Replace this with your actual Runge function setup from Project 1.
# Here we simulate a 1D feature input (X) and continuous target (y)
def runge_function(x):
    # f(x) = 1 / (1 + 25*x^2)
    return 1.0 / (1.0 + 25 * x**2) 

# Generate data points
N = 100
x = np.linspace(-1, 1, N).reshape(-1, 1)
y = runge_function(x)

# Split and Scale data (required for Part b) [3]
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# --- Network Training Configuration ---

# Required dimensions: Input (1 feature), 1 hidden layer (50 nodes), Output (1 regression value) [14]
input_nodes = X_train_scaled.shape[1]
hidden_nodes1 = 50
output_nodes = 1

dims = (input_nodes, hidden_nodes1, output_nodes)

# Initialize NN for regression (sigmoid hidden, linear output, MSE cost)
nn_regressor = FFNN(
    dims, 
    hidden_func=sigmoid, 
    output_func=identity, 
    cost_func=CostOLS, 
    seed=42
)

# Choose an optimizer (e.g., ADAM, required in Part b) [3]
scheduler = Adam(eta=0.01, rho=0.9, rho2=0.999) 

print(f"Starting training with {scheduler.__class__.__name__} optimizer:")
nn_regressor.fit(
    X_train_scaled, 
    y_train, 
    scheduler, 
    batches=10, 
    epochs=1000, 
    lam=0.0  # No regularization initially (Part b requirement)
)

# Evaluate results
y_pred_test = nn_regressor.predict(X_test_scaled)
mse_final = CostOLS(y_test)(y_pred_test)
print(f"\nFinal Test MSE: {mse_final:.6f}")

TypeError: NeuralNetwork.__init__() got an unexpected keyword argument 'hidden_func'